# Rebuild `smap_by_sample.parquet` from the six GEE CSV exports

This notebook merges the six CSV files in `Data/Global/smap/` and attaches their SMAP values to `station_date_candidates.parquet`.

**Important schema detail:** `station_date_candidates.parquet` has `station_id`, not `station_key`. This notebook uses `station_smap_grid_mapping.parquet` to translate `station_id` to the `station_key` used by the GEE CSV exports.

Run the cells from top to bottom. Review the matching diagnostics before writing the output. The original `smap_by_sample.parquet`, if present, is backed up before it is replaced.


In [1]:
from pathlib import Path
import shutil
import pandas as pd
import numpy as np

# If this notebook is saved in Data/Global, this default works.
# Otherwise change PROJECT_DIR to the absolute path of your Data/Global directory.
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != "Global" or not (PROJECT_DIR / "station_date_candidates.parquet").exists():
    candidates = [
        Path.home() / "Code/UG_SM/Data/Global",
        Path.cwd() / "Data/Global",
    ]
    PROJECT_DIR = next((p for p in candidates if (p / "station_date_candidates.parquet").exists()), PROJECT_DIR)

SMAP_DIR = PROJECT_DIR / "smap"
SAMPLES_PATH = PROJECT_DIR / "station_date_candidates.parquet"
MAPPING_PATH = SMAP_DIR / "station_smap_grid_mapping.parquet"
OUTPUT_PATH = SMAP_DIR / "smap_by_sample.parquet"

print("Project directory:", PROJECT_DIR.resolve())
print("CSV directory:", SMAP_DIR.resolve())
print("Candidates exist:", SAMPLES_PATH.exists())
print("Mapping exists:", MAPPING_PATH.exists())


Project directory: /home/mayur/Code/UG_SM/Data/Global
CSV directory: /home/mayur/Code/UG_SM/Data/Global/smap
Candidates exist: True
Mapping exists: True


In [2]:
# Find and inspect the CSV exports.
csv_files = sorted(SMAP_DIR.glob("*.csv"))
print(f"CSV files found: {len(csv_files)}")
for p in csv_files:
    print(f"  {p.name} ({p.stat().st_size:,} bytes)")

if len(csv_files) != 6:
    raise ValueError(f"Expected exactly 6 CSV files in {SMAP_DIR}, found {len(csv_files)}.")

csv_preview = []
for path in csv_files:
    preview = pd.read_csv(path, nrows=3)
    csv_preview.append({
        "file": path.name,
        "columns": list(preview.columns),
        "preview_rows": len(preview)
    })
pd.DataFrame(csv_preview)


CSV files found: 6
  SMAP_L4_DAILY_2015_2020_B01.csv (195,840,307 bytes)
  SMAP_L4_DAILY_2015_2020_B02.csv (200,990,692 bytes)
  SMAP_L4_DAILY_2015_2020_B03.csv (41,550,431 bytes)
  SMAP_L4_DAILY_2020_end_B01.csv (255,156,238 bytes)
  SMAP_L4_DAILY_2020_end_B02.csv (261,880,382 bytes)
  SMAP_L4_DAILY_2020_end_B03.csv (54,139,516 bytes)


,file,columns,preview_rows
0,SMAP_L4_DAILY_2015_2020_B01.csv,"[system:index, date, sm_surface, smap_n, stati...",3
1,SMAP_L4_DAILY_2015_2020_B02.csv,"[system:index, date, sm_surface, smap_n, stati...",3
2,SMAP_L4_DAILY_2015_2020_B03.csv,"[system:index, date, sm_surface, smap_n, stati...",3
3,SMAP_L4_DAILY_2020_end_B01.csv,"[system:index, date, sm_surface, smap_n, stati...",3
4,SMAP_L4_DAILY_2020_end_B02.csv,"[system:index, date, sm_surface, smap_n, stati...",3
5,SMAP_L4_DAILY_2020_end_B03.csv,"[system:index, date, sm_surface, smap_n, stati...",3


In [3]:
# Load the canonical sample table and station-to-SMAP-key mapping.
samples = pd.read_parquet(SAMPLES_PATH)
mapping = pd.read_parquet(MAPPING_PATH)

print("Sample table shape:", samples.shape)
print("Sample columns:", samples.columns.tolist())
print("Mapping shape:", mapping.shape)
print("Mapping columns:", mapping.columns.tolist())

required_samples = {"sample_id", "station_id", "sample_date"}
required_mapping = {"station_id", "station_key"}
if not required_samples.issubset(samples.columns):
    raise ValueError(f"Sample table is missing required columns: {sorted(required_samples - set(samples.columns))}")
if not required_mapping.issubset(mapping.columns):
    raise ValueError(f"Mapping table is missing required columns: {sorted(required_mapping - set(mapping.columns))}")

display(samples.head())
display(mapping[["station_id", "station_key"]].head())


Sample table shape: (3514643, 15)
Sample columns: ['sample_id', 'station_id', 'sample_date', 'network', 'network_name', 'station', 'latitude', 'longitude', 'elevation_m', 'climate_zone', 'source_row_count', 'min_depth_from_m', 'max_depth_to_m', 'sensors', 'depth_classes']
Mapping shape: (2210, 17)
Mapping columns: ['station_index', 'station_id', 'network', 'network_name', 'station', 'latitude', 'longitude', 'elevation_m', 'climate_zone', 'station_key', 'smap_x_m', 'smap_y_m', 'smap_x_idx', 'smap_y_idx', 'smap_cell_x_m', 'smap_cell_y_m', 'smap_distance_m']


,sample_id,station_id,sample_date,network,network_name,station,latitude,longitude,elevation_m,climate_zone,source_row_count,min_depth_from_m,max_depth_to_m,sensors,depth_classes
0,0,AMMA-CATCH__Banizoumbou,2015-04-01,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,BSh,2,0.05,0.05,Campbell-CS616,near_surface
1,1,AMMA-CATCH__Banizoumbou,2015-04-02,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,BSh,2,0.05,0.05,Campbell-CS616,near_surface
2,2,AMMA-CATCH__Banizoumbou,2015-04-03,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,BSh,2,0.05,0.05,Campbell-CS616,near_surface
3,3,AMMA-CATCH__Banizoumbou,2015-04-04,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,BSh,2,0.05,0.05,Campbell-CS616,near_surface
4,4,AMMA-CATCH__Banizoumbou,2015-04-05,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,BSh,2,0.05,0.05,Campbell-CS616,near_surface


,station_id,station_key
0,AMMA-CATCH__Banizoumbou,AMMA-CATCH::AMMA-CATCH__Banizoumbou
1,AMMA-CATCH__Belefoungou-Mid,AMMA-CATCH::AMMA-CATCH__Belefoungou-Mid
2,AMMA-CATCH__Belefoungou-Top,AMMA-CATCH::AMMA-CATCH__Belefoungou-Top
3,AMMA-CATCH__Nalohou-Mid,AMMA-CATCH::AMMA-CATCH__Nalohou-Mid
4,AMMA-CATCH__Nalohou-Top,AMMA-CATCH::AMMA-CATCH__Nalohou-Top


In [4]:
# Normalize IDs and build a safe station_id -> station_key lookup.
samples["station_id"] = samples["station_id"].astype("string").str.strip()
mapping["station_id"] = mapping["station_id"].astype("string").str.strip()
mapping["station_key"] = mapping["station_key"].astype("string").str.strip()

station_lookup = mapping[["station_id", "station_key"]].dropna().drop_duplicates()

# A station_id must not map to multiple distinct station_keys.
key_counts = station_lookup.groupby("station_id")["station_key"].nunique()
ambiguous_ids = key_counts[key_counts > 1]
if not ambiguous_ids.empty:
    display(station_lookup[station_lookup["station_id"].isin(ambiguous_ids.index)].sort_values("station_id").head(30))
    raise ValueError(f"{len(ambiguous_ids)} station_id values map to multiple station_keys. Resolve these before merging.")

station_lookup = station_lookup.drop_duplicates("station_id")
samples = samples.merge(station_lookup, on="station_id", how="left", validate="many_to_one")

mapped_share = samples["station_key"].notna().mean()
print(f"Sample rows with a station_key mapping: {mapped_share:.2%} ({samples['station_key'].notna().sum():,}/{len(samples):,})")
unmapped = samples.loc[samples["station_key"].isna(), ["station_id", "network", "network_name", "station"]].drop_duplicates()
print("Unique unmapped stations:", len(unmapped))
display(unmapped.head(20))

if mapped_share < 0.95:
    raise ValueError("Fewer than 95% of sample rows mapped to station_key. Check station IDs and mapping table before continuing.")


Sample rows with a station_key mapping: 100.00% (3,514,643/3,514,643)
Unique unmapped stations: 0


,station_id,network,network_name,station


In [5]:
# Read the six GEE exports, keeping only fields needed for the join.
parts = []
needed_csv_columns = {"date", "sm_surface", "smap_n", "station_key"}

for path in csv_files:
    # Read only relevant columns; ignore system:index and _geo.
    header = pd.read_csv(path, nrows=0).columns.tolist()
    missing = needed_csv_columns - set(header)
    if missing:
        raise ValueError(f"{path.name} is missing required columns: {sorted(missing)}")

    frame = pd.read_csv(path, usecols=["date", "sm_surface", "smap_n", "station_key"])
    frame["station_key"] = frame["station_key"].astype("string").str.strip()
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce", utc=True).dt.tz_convert(None).dt.normalize()
    frame["sm_surface"] = pd.to_numeric(frame["sm_surface"], errors="coerce")
    frame["smap_n"] = pd.to_numeric(frame["smap_n"], errors="coerce")
    frame["source_file"] = path.name
    parts.append(frame)

smap = pd.concat(parts, ignore_index=True)
print("Raw SMAP rows:", f"{len(smap):,}")
print("Invalid dates:", f"{smap['date'].isna().sum():,}")
print("Missing station_key:", f"{smap['station_key'].isna().sum():,}")

smap = smap.dropna(subset=["station_key", "date"]).copy()
smap = smap.drop_duplicates(subset=["station_key", "date", "sm_surface", "smap_n"])
print("Rows after dropping exact duplicates:", f"{len(smap):,}")
display(smap.head())


Raw SMAP rows: 9,286,636
Invalid dates: 0
Missing station_key: 0
Rows after dropping exact duplicates: 9,286,636


,date,sm_surface,smap_n,station_key,source_file
0,2015-04-01,0.022373,8,AMMA-CATCH::AMMA-CATCH__Banizoumbou,SMAP_L4_DAILY_2015_2020_B01.csv
1,2015-04-01,0.140224,8,AMMA-CATCH::AMMA-CATCH__Belefoungou-Mid,SMAP_L4_DAILY_2015_2020_B01.csv
2,2015-04-01,0.140224,8,AMMA-CATCH::AMMA-CATCH__Belefoungou-Top,SMAP_L4_DAILY_2015_2020_B01.csv
3,2015-04-01,0.140007,8,AMMA-CATCH::AMMA-CATCH__Nalohou-Mid,SMAP_L4_DAILY_2015_2020_B01.csv
4,2015-04-01,0.140007,8,AMMA-CATCH::AMMA-CATCH__Nalohou-Top,SMAP_L4_DAILY_2015_2020_B01.csv


In [ ]:
# Check whether the exports contain conflicting values for the same station and date.
value_counts = (
    smap.groupby(["station_key", "date"], dropna=False)[["sm_surface", "smap_n"]]
    .nunique(dropna=False)
)
conflicts = value_counts[(value_counts["sm_surface"] > 1) | (value_counts["smap_n"] > 1)]

print("Unique station/date pairs:", f"{len(value_counts):,}")
print("Conflicting station/date pairs:", f"{len(conflicts):,}")

if not conflicts.empty:
    conflict_keys = conflicts.reset_index()[["station_key", "date"]]
    conflict_rows = smap.merge(conflict_keys, on=["station_key", "date"], how="inner")
    display(conflict_rows.sort_values(["station_key", "date"]).head(50))
    raise ValueError(
        "Conflicting SMAP values exist for the same station_key/date. "
        "Do not write output until the duplicate/conflict source is understood."
    )

smap_daily = smap.drop_duplicates(["station_key", "date"]).copy()
print("Unique daily SMAP rows:", f"{len(smap_daily):,}")


Unique station/date pairs: 9,286,636
Conflicting station/date pairs: 0
Unique daily SMAP rows: 9,286,636


: 

In [ ]:
# Normalize sample dates and join SMAP values to each sample.
samples["sample_date"] = pd.to_datetime(samples["sample_date"], errors="coerce", utc=True).dt.tz_convert(None).dt.normalize()
if samples["sample_date"].isna().any():
    raise ValueError(f"{samples['sample_date'].isna().sum()} sample rows have invalid sample_date values.")

if samples["sample_id"].duplicated().any():
    raise ValueError("sample_id is not unique in station_date_candidates.parquet.")

merged = samples.merge(
    smap_daily[["station_key", "date", "sm_surface", "smap_n"]],
    left_on=["station_key", "sample_date"],
    right_on=["station_key", "date"],
    how="left",
    validate="many_to_one",
    indicator=True,
)

matched_rows = int((merged["_merge"] == "both").sum())
print(f"Matched samples: {matched_rows:,} / {len(samples):,} ({matched_rows / max(len(samples), 1):.2%})")
print("Samples without SMAP match:", f"{(merged['_merge'] == 'left_only').sum():,}")

# Keep the original sample columns and append the standardized output columns.
original_columns = [c for c in samples.columns if c != "date"]
result = merged[original_columns].copy()
result["SMAP"] = merged["sm_surface"]
result["SMAP_n_obs"] = merged["smap_n"].astype("Int64")
display(result.head())


In [ ]:
# Write to a temporary Parquet, validate it, then atomically replace the target.
# Existing output is preserved as a timestamped backup.
from datetime import datetime

SMAP_DIR.mkdir(parents=True, exist_ok=True)
temp_path = OUTPUT_PATH.with_name(OUTPUT_PATH.stem + ".tmp.parquet")

if OUTPUT_PATH.exists():
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = OUTPUT_PATH.with_name(f"{OUTPUT_PATH.stem}.backup_{stamp}.parquet")
    shutil.copy2(OUTPUT_PATH, backup_path)
    print("Existing output backed up to:", backup_path)

result.to_parquet(temp_path, index=False)

# Re-read and validate the temporary output before replacing anything.
check = pd.read_parquet(temp_path)
assert len(check) == len(samples), f"Row count changed: {len(check)} vs {len(samples)}"
assert check["sample_id"].is_unique, "sample_id is not unique in output"
assert {"SMAP", "SMAP_n_obs"}.issubset(check.columns), "Expected SMAP columns missing"

temp_path.replace(OUTPUT_PATH)
print("Wrote:", OUTPUT_PATH.resolve())
print("Output rows:", f"{len(check):,}")
print("Rows with SMAP:", f"{check['SMAP'].notna().sum():,}")
print("Rows without SMAP:", f"{check['SMAP'].isna().sum():,}")
print("Output size:", f"{OUTPUT_PATH.stat().st_size:,} bytes")


## Notes

- `SMAP` is copied from the GEE CSV column `sm_surface`.
- `SMAP_n_obs` is copied from `smap_n`.
- `system:index` and `_geo` are ignored because the join uses `station_key` plus calendar date.
- The output preserves the rows and original columns from `station_date_candidates.parquet`, with the two standardized SMAP columns appended.
- If the station mapping check fails, inspect the displayed unmapped stations before proceeding; do not weaken the threshold blindly.
